# IMDb movie analysis

Tasks A–E from the brief. Pandas first, then the same headlines in SQLite.
Excel workbook is `excel/imdb_analysis.xlsx` (includes `=CORREL`).


In [1]:
from pathlib import Path
import sqlite3
import pandas as pd
import numpy as np

ROOT = None
for cand in [Path(".."), Path("."), Path("/workspace/artifacts/imdb-movie-analysis")]:
    if (cand / "data" / "imdb_movies.csv").exists():
        ROOT = cand
        break
df = pd.read_csv(ROOT / "data" / "imdb_movies.csv")
print(df.shape, "mean score", round(df.imdb_score.mean(), 3), "years", int(df.title_year.min()), int(df.title_year.max()))


(4916, 28) mean score 6.437 years 1916 2016


## A. Genre

In [2]:
g = df.dropna(subset=["genres"]).copy()
g["genre"] = g["genres"].str.split("|")
g = g.explode("genre")
g["genre"] = g["genre"].str.strip()
stats = (g.groupby("genre")["imdb_score"]
         .agg(n="count", mean="mean", median="median",
              mode=lambda s: s.mode().iloc[0],
              min="min", max="max",
              rng=lambda s: s.max()-s.min(), var="var", std="std")
         .sort_values("n", ascending=False)
         .round(3))
display(stats.head(12))
print("highest mean n>=50")
display(stats[stats.n>=50].sort_values("mean", ascending=False).head(5))


,n,mean,median,mode,min,max,rng,var,std
genre,,,,,,,,,
Drama,2532,6.765,6.9,6.7,2.0,9.3,7.3,0.910,0.954
Comedy,1847,6.192,6.3,6.7,1.7,9.5,7.8,1.191,1.091
Thriller,1364,6.308,6.4,6.4,2.2,9.0,6.8,1.116,1.056
Action,1113,6.232,6.3,6.1,1.7,9.1,7.4,1.253,1.119
Romance,1084,6.447,6.5,6.5,2.1,8.6,6.5,0.997,0.998
Adventure,888,6.439,6.6,6.7,1.9,8.9,7.0,1.294,1.137
Crime,869,6.563,6.6,6.3,2.4,9.3,6.9,1.059,1.029
Sci-Fi,593,6.277,6.4,6.7,1.9,8.8,6.9,1.482,1.217
Fantasy,582,6.301,6.4,6.7,1.7,8.9,7.2,1.362,1.167


highest mean n>=50


,n,mean,median,mode,min,max,rng,var,std
genre,,,,,,,,,
Documentary,121,7.180,7.4,7.5,1.6,8.7,7.1,1.116,1.057
Biography,291,7.149,7.2,7.0,4.5,8.9,4.4,0.525,0.725
History,202,7.091,7.2,7.5,2.0,8.9,6.9,0.786,0.886
War,210,7.071,7.1,7.1,2.7,8.6,5.9,0.767,0.876
Drama,2532,6.765,6.9,6.7,2.0,9.3,7.3,0.910,0.954


## B. Duration

In [3]:
d = df.dropna(subset=["duration","imdb_score"])
print(d.duration.describe()[["mean","50%","std","min","max"]].round(2).to_dict())
print("mode", d.duration.mode().tolist(), "r", round(d.duration.corr(d.imdb_score), 3))
d = d.copy()
d["bin"] = pd.cut(d.duration, [0,90,120,150,999], labels=["<90","90-120","120-150",">150"])
display(d.groupby("bin", observed=True)["imdb_score"].agg(["count","mean","median"]).round(3))


{'mean': 107.1, '50%': 103.0, 'std': 25.28, 'min': 7.0, 'max': 511.0}
mode [90.0] r 0.265


,count,mean,median
bin,,,
<90,940,6.114,6.2
90-120,2922,6.322,6.4
120-150,839,6.965,7.0
>150,200,7.443,7.6


## C. Language

In [4]:
lang = (df.dropna(subset=["language"])
          .groupby("language")["imdb_score"]
          .agg(n="count", mean="mean", median="median", std="std")
          .sort_values("n", ascending=False)
          .round(3))
display(lang.head(10))
print("highest mean n>=10")
display(lang[lang.n>=10].sort_values("mean", ascending=False).head(6))


,n,mean,median,std
language,,,,
English,4582,6.393,6.50,1.125
French,73,7.038,7.20,0.727
Spanish,40,6.938,7.15,0.855
Hindi,28,6.632,6.95,1.399
Mandarin,24,6.788,7.05,1.037
German,19,7.342,7.60,0.954
Japanese,17,7.347,7.50,1.000
Cantonese,11,6.955,7.20,0.705
Italian,11,7.227,7.30,1.244


highest mean n>=10


,n,mean,median,std
language,,,,
Japanese,17,7.347,7.50,1.000
German,19,7.342,7.60,0.954
Italian,11,7.227,7.30,1.244
French,73,7.038,7.20,0.727
Cantonese,11,6.955,7.20,0.705
Spanish,40,6.938,7.15,0.855


## D. Directors (≥ 5 films)

In [5]:
ds = (df.dropna(subset=["director_name"])
        .groupby("director_name")["imdb_score"]
        .agg(n="count", mean="mean", median="median")
        .query("n >= 5")
        .sort_values("mean", ascending=False))
ds["pct"] = (ds["mean"].rank(pct=True)*100).round(1)
display(ds.head(12).round(3))
print("n directors", len(ds), "p90", round(np.percentile(ds["mean"], 90), 3))


,n,mean,median,pct
director_name,,,,
Christopher Nolan,8,8.425,8.50,100.0
Quentin Tarantino,8,8.200,8.20,99.5
Frank Capra,5,8.060,8.20,99.1
Stanley Kubrick,6,8.050,8.20,98.6
James Cameron,7,7.914,7.90,98.1
Peter Jackson,9,7.889,7.90,97.7
Alejandro G. Iñárritu,6,7.783,7.75,97.2
Fred Zinnemann,5,7.760,7.80,96.7
David Fincher,10,7.750,7.80,96.3


n directors 214 p90 7.412


## E. Budget vs gross

In [6]:
b = df.dropna(subset=["budget","gross"]).copy()
b["profit"] = b.gross - b.budget
print("raw r", round(b.budget.corr(b.gross), 3), "n", len(b))
sane = b[b.budget <= 400_000_000]
print("sane r", round(sane.budget.corr(sane.gross), 3), "n", len(sane),
      "vs score", round(sane.budget.corr(sane.imdb_score), 3))
display(sane.nlargest(8, "profit")[["movie_title","budget","gross","profit","imdb_score"]])
print("FX suspects")
display(b[b.budget>400_000_000][["movie_title","country","budget","gross"]].head(6))


raw r 0.223 n 3789
sane r 0.627 n 3779 vs score 0.036


,movie_title,budget,gross,profit,imdb_score
20,Avatar,237000000.0,760505847.0,523505847.0,7.9
158,Jurassic World,150000000.0,652177271.0,502177271.0,7.0
30,Titanic,200000000.0,658672302.0,458672302.0,7.7
18,Star Wars: Episode IV - A New Hope,11000000.0,460935665.0,449935665.0,8.7
326,E.T. the Extra-Terrestrial,10500000.0,434949459.0,424449459.0,7.9
13,The Avengers,220000000.0,623279547.0,403279547.0,8.1
57,The Lion King,45000000.0,422783777.0,377783777.0,8.5
87,Star Wars: Episode I - The Phantom Menace,115000000.0,474544677.0,359544677.0,6.5


FX suspects


,movie_title,country,budget,gross
463,Princess Mononoke,Japan,2.400000e+09,2298191.0
1102,Akira,Japan,1.100000e+09,439162.0
1935,Lady Vengeance,South Korea,4.200000e+09,211667.0
2340,Red Cliff,China,5.536320e+08,626809.0
3301,Kabhi Alvida Naa Kehna,India,7.000000e+08,3275443.0
3312,Steamboy,Japan,2.127520e+09,410388.0


## Same headlines in SQLite

In [7]:
con = sqlite3.connect(":memory:")
df.to_sql("movies", con, index=False, if_exists="replace")
g[["movie_title","genre","imdb_score"]].to_sql("movie_genres", con, index=False, if_exists="replace")
sane.to_sql("sane_budget", con, index=False, if_exists="replace")
print("top genres")
display(pd.read_sql("SELECT genre, COUNT(*) n, ROUND(AVG(imdb_score),3) mean_score FROM movie_genres GROUP BY genre ORDER BY n DESC LIMIT 5", con))
print("Nolan")
display(pd.read_sql("SELECT director_name, COUNT(*) n, ROUND(AVG(imdb_score),3) mean_score FROM movies WHERE director_name IS NOT NULL GROUP BY director_name HAVING COUNT(*)>=5 ORDER BY mean_score DESC LIMIT 5", con))
print("top profit")
display(pd.read_sql("SELECT movie_title, (gross-budget) profit FROM sane_budget ORDER BY profit DESC LIMIT 3", con))
con.close()


top genres


,genre,n,mean_score
0,Drama,2532,6.765
1,Comedy,1847,6.192
2,Thriller,1364,6.308
3,Action,1113,6.232
4,Romance,1084,6.447


Nolan


,director_name,n,mean_score
0,Christopher Nolan,8,8.425
1,Quentin Tarantino,8,8.200
2,Frank Capra,5,8.060
3,Stanley Kubrick,6,8.050
4,James Cameron,7,7.914


top profit


,movie_title,profit
0,Avatar,523505847.0
1,Jurassic World,502177271.0
2,Titanic,458672302.0


## Takeaways

- Documentary / biography score higher; horror lower. Drama is the volume tag.
- Runtime r = 0.26 with IMDb. Budget r ≈ 0.04 with IMDb, 0.63 with gross (USD-ish).
- Rank directors on ≥5 films. Nolan 8.425.
- Excel `=CORREL` is on sheet E_correl.
